# HisToGene Patch-H5 Training Smoke

This notebook runs a tiny HEST coverage95 HisToGene-style training smoke test. It verifies that the patch-H5 chunk model can train, save a checkpoint, and export benchmark-shaped `log1p_rate` prediction arrays. The output is not a formal baseline result.

In [1]:
import contextlib
import io
import json
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "histoomnist").exists() and (ROOT.parent / "src" / "histoomnist").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

warnings.filterwarnings("ignore", message="enable_nested_tensor is True.*")

import numpy as np

from histoomnist.external.train_histogene_patch import export_histogene_patch_predictions, train_histogene_patch
from histoomnist.utils.config import load_config
from histoomnist.utils.project_paths import resolve_project_path

In [2]:
cfg = load_config(resolve_project_path("configs/hest1k_human_visium_expression_highconf_symbol95.yaml"))
model_cfg = {"patch_size": 28, "dim": 32, "n_layers": 1, "n_heads": 4, "dropout": 0.1, "n_pos": 64}
out_dir = resolve_project_path("checkpoints/hest1k_human_visium_expression_external/histogene_patch_h5_smoke")
pred_root = resolve_project_path("results/hest1k_human_visium_expression/external_baselines/histogene_patch_h5_train_smoke_predictions")

with contextlib.redirect_stdout(io.StringIO()):
    train_summary = train_histogene_patch(
        expression_config=cfg,
        train_splits=["train"],
        val_splits=["val"],
        output_dir=out_dir,
        model_cfg=model_cfg,
        target_kind="log1p_rate",
        epochs=1,
        batch_size=1,
        chunk_size=4,
        lr=1.0e-4,
        weight_decay=0.0,
        max_train_slides=1,
        max_val_slides=1,
        max_train_chunks_per_slide=1,
        max_val_chunks_per_slide=1,
    )
    prediction_summary = export_histogene_patch_predictions(
        expression_config=cfg,
        checkpoint_path=train_summary["checkpoint"],
        out_dir=pred_root,
        splits=["test"],
        batch_size=1,
        chunk_size=4,
        max_slides=1,
        max_chunks_per_slide=1,
        max_spots_per_slide=4,
    )

sanitized = {
    "device": train_summary["device"],
    "epochs": train_summary["epochs"],
    "chunk_size": train_summary["chunk_size"],
    "n_train_chunks": train_summary["n_train_chunks"],
    "n_val_chunks": train_summary["n_val_chunks"],
    "n_genes": train_summary["n_genes"],
    "best_val_loss": train_summary["best_val_loss"],
    "prediction_kind": prediction_summary["prediction_kind"],
    "predicted_slides": prediction_summary["slides"],
}
sanitized

{'device': 'cuda',
 'epochs': 1,
 'chunk_size': 4,
 'n_train_chunks': 1,
 'n_val_chunks': 1,
 'n_genes': 16942,
 'best_val_loss': 0.8827410340309143,
 'prediction_kind': 'log1p_rate',
 'predicted_slides': [{'sample_id': 'MISC33',
   'n_predicted_spots': 4,
   'expected_spots': 3385,
   'complete_slide_prediction': False}]}

In [3]:
genes = (pred_root / "genes.txt").read_text(encoding="utf-8").splitlines()
pred_path = pred_root / "predictions" / "MISC33_log1p_rate.npy"
pred = np.load(pred_path, allow_pickle=False)
checks = {
    "genes": len(genes),
    "prediction_shape": tuple(pred.shape),
    "prediction_finite": bool(np.isfinite(pred).all()),
    "prediction_std": float(pred.std()),
}
assert checks["genes"] == 16942
assert checks["prediction_shape"] == (4, 16942)
assert checks["prediction_finite"]
checks

{'genes': 16942,
 'prediction_shape': (4, 16942),
 'prediction_finite': True,
 'prediction_std': 0.5846068859100342}